In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#In this code I am going to implement the advantage actor-critic (A2C) algorithm, with distance-based reward shaping
#Jeremy Kedziora
#4 November 2026
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@
#import useful libraries
#@@@@@@@@@@@@@@@@@@@@@@@
import numpy as np

from tqdm import tqdm
from IPython.display import clear_output
import time
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
class Chase_environment:
    '''A class to manage the chase'''
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def __init__(self, config):
        '''Set up the constructor
            Takes -- config, a dictionary specifying the grid size, start locations, predator radius, prey max speed, max turns per episode, and the discount factor, and the reward shaping weights
        '''
        self.config = config
        self.max_distance = (2*(self.config['grid_size'] - 1)**2)**0.5    #the longest possible distance in the arena (corner to corner), used to scale the shaping terms
        self.reset()    #set everything up
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def reset(self):
        '''A function to put the predator and prey at their starting locations and stop the prey
        '''
        self.t = 0    #the number of turns taken so far this episode
        self.state = {}
        if self.config['predator_start']:
            self.state['predator_location'] = dict(self.config['predator_start'])    #copy so the moves don't overwrite the config
        else:
            self.state['predator_location'] = {'X':np.random.choice(a = range(1,self.config['grid_size']))
                                                  ,'Y':np.random.choice(a = range(1,self.config['grid_size']))}
        if self.config['prey_start']:
            self.state['prey_location'] = dict(self.config['prey_start'])    #copy so the moves don't overwrite the config
        else:
            self.state['prey_location'] = {'X':0,'Y':0}
        self.state['prey_V'] = {'V':0,'H':0}

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def get_state(self):
        '''A function to package the current state for the agent
            Returns -- a list with the predator X and Y, the prey X and Y, and the prey horiz and vert velocities
        '''
        return [self.state['predator_location']['X']
                ,self.state['predator_location']['Y']
                ,self.state['prey_location']['X']
                ,self.state['prey_location']['Y']
                ,self.state['prey_V']['H']
                ,self.state['prey_V']['V']]

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@    
    def distance(self, x1, x2):    
        '''A function to compute the Euclidean distance between two points
            Takes -- x1, x2, the [X,Y] coordinates of the two points
        '''
        return ((x1[0] - x2[0])**2 + (x1[1] - x2[1])**2)**0.5
        
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@    
    def update_V(self, a):
        '''A function to update vert and horiz velocity
            Takes -- a, a dictionary containing the vert and horiz acceleration
        '''
        if a['V'] == 1:
            a_v = np.random.uniform(low = 0.5,high = 1.5)
        elif a['V'] == 0:
            a_v = 0
        elif a['V'] == -1:
            a_v = np.random.uniform(low = -1.5,high = -0.5)
        else:
            raise ValueError(f"vertical acceleration must be -1, 0, or 1, not {a['V']}")
        if a['H'] == 1:
            a_h = np.random.uniform(low = 0.5,high = 1.5)
        elif a['H'] == 0:
            a_h = 0
        elif a['H'] == -1:
            a_h = np.random.uniform(low = -1.5,high = -0.5)
        else:
            raise ValueError(f"horizontal acceleration must be -1, 0, or 1, not {a['H']}")
        v = self.state['prey_V']['V'] + a_v    #the new vert velocity
        h = self.state['prey_V']['H'] + a_h    #the new horiz velocity
        speed = (v**2 + h**2)**0.5    #how fast the prey is going
        if speed > self.config['max_speed']:    #if the prey is going too fast...
            v = v*self.config['max_speed']/speed    #shrink the velocity back onto the circle of radius max_speed, keeping its direction
            h = h*self.config['max_speed']/speed
        self.state['prey_V']['V'] = v    #update the vert velocity
        self.state['prey_V']['H'] = h    #update the horiz velocity

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def update_Position(self):
        '''A function to update vert and horiz position
        '''        
        new_position_X = min(max(self.state['prey_location']['X'] + self.state['prey_V']['H'],0),self.config['grid_size'] - 1)
        new_position_Y = min(max(self.state['prey_location']['Y'] + self.state['prey_V']['V'],0),self.config['grid_size'] - 1)            
        self.state['prey_location']['X'] = new_position_X    #update the horiz position
        self.state['prey_location']['Y'] = new_position_Y    #update the vert position
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@    
    def update_Predator(self):
        '''A function to move the predator toward the prey
        '''
        pred = np.array([self.state['predator_location']['X'],self.state['predator_location']['Y']])    #the predator position
        prey = np.array([self.state['prey_location']['X'],self.state['prey_location']['Y']])    #the prey position
        d = self.distance(pred,prey)    #how far apart they are
        if d <= self.config['predator_radius']:    #if the prey is within reach...
            new_position = prey    #the predator lands on the prey
        else:    #otherwise...
            new_position = pred + self.config['predator_radius']*(prey - pred)/d    #move one radius along the line toward the prey
        self.state['predator_location']['X'] = float(new_position[0])    #update the horiz position
        self.state['predator_location']['Y'] = float(new_position[1])    #update the vert position

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@    
    def check_caught(self):
        '''A function to check if the prey is caught'''
        caught = False
        if self.state['predator_location']['X'] == self.state['prey_location']['X'] and self.state['predator_location']['Y'] == self.state['prey_location']['Y']:
             caught = True
        return caught
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@    
    def check_base(self):
        '''A function to check in the prey in on base'''
        base = False
        if self.state['prey_location']['X'] == self.config['grid_size'] - 1 and self.state['prey_location']['Y'] == self.config['grid_size'] - 1:
            base = True
        return base

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def potential(self):
        '''A function to compute the shaping potential Phi of the current state
            Returns -- float, a potential that is large when the prey is close to base or far from the predator, and small when the prey is far from base or close to the predator
        '''
        prey = [self.state['prey_location']['X'],self.state['prey_location']['Y']]    #the prey position
        pred = [self.state['predator_location']['X'],self.state['predator_location']['Y']]    #the predator position
        base = [self.config['grid_size'] - 1,self.config['grid_size'] - 1]    #the base position
        d_base = self.distance(prey,base)/self.max_distance    #how far the prey is from base, between 0 and 1
        d_pred = self.distance(prey,pred)/self.max_distance    #how far the prey is from the predator, between 0 and 1
        return self.config['base_weight']*(1 - d_base) + self.config['predator_weight']*d_pred    #high potential near base and far from the predator

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@    
    def step(self, a):
        '''A function to evolve the system given an action
            Returns -- a dictionary with the state, the shaped reward, whether the episode is done, and whether the prey won
        '''
        phi_s = self.potential()    #the potential of the state before the move
        self.t += 1    #count the turn
        win = False
        lost = False
        if np.random.uniform() < 0.5:    #if the prey moves first...
            self.update_V(a)    #update the velocity
            self.update_Position()    #update the position
            win = self.check_base()    #see if the prey reached base
            if not win:    #if it didn't...
                self.update_Predator()    #update the predator location
                lost = self.check_caught()    #check to see if the predator caught the prey
        else:    #if the predator moves first...
            self.update_Predator()    #update the predator location
            lost = self.check_caught()    #check to see if the predator caught the prey
            if not lost:    #it if didn't...
                self.update_V(a)    #update prey velocity
                self.update_Position()    #update prey location
                win = self.check_base()    #see if prey reached base
        terminated = win or lost    #a win or a capture ends the chase for good
        truncated = self.t >= self.config['max_steps']    #running out of turns only stops the clock -- the agent still treats the state as having a future
        done = terminated or truncated
        #potential-based shaping: F = gamma*Phi(s') - Phi(s), with Phi = 0 at terminal states
        #because the shaping terms telescope along any trajectory, this changes how fast the agent learns but not which policy is optimal (Ng, Harada & Russell, 1999)
        phi_s_next = 0.0 if terminated else self.potential()    #a terminal state has no future, so no potential
        r = self.config['gamma']*phi_s_next - phi_s    #reward moving toward base and away from the predator, penalize the reverse
        if win:
            r += 10    #the reward for reaching base
        return {'state':self.state,'reward':r,'done':done,'win':win}


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
# Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)    #First fully connected layer
        self.fc2 = nn.Linear(hidden_size, hidden_size)    #Second fully connected layer
        self.fc3 = nn.Linear(hidden_size, output_size)    #Output layer, one logit per action for the actor or a single state value for the critic

    def forward(self, x):
        x = F.relu(self.fc1(x))    #ReLU activation for the first layer
        x = F.relu(self.fc2(x))    #ReLU activation for the second layer
        x = self.fc3(x)    #Output layer (no activation here -- the actor's logits become action probabilities through a softmax, and the critic's value can be any real number)
        return x
    

In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
class Agent:
    '''A class to manage the A2C agent (the prey)'''
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def __init__(self, config):
        '''Set up the constructor
            Takes -- config, a dictionary specifying the agent's hyperparameters and action set
        '''
        self.config = config
        self.scale = torch.tensor([self.config['grid_size'] - 1]*4 + [self.config['max_speed']]*2).to(torch.float32)    #divide positions by the largest position and velocities by the max speed so positions are between 0 and 1 and velocities between -1 and 1
        self.reset()    #build the actor and critic

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def reset(self):
        '''A function to reset the actor and critic MLPs to random initial parameters'''
        self.pi_net = MLP(input_size=6, hidden_size=self.config['hidden_size'], output_size=len(self.config['A']))    #the actor: the state goes in, a logit for every action comes out
        self.V_net = MLP(input_size=6, hidden_size=self.config['hidden_size'], output_size=1)    #the critic: the state goes in, an estimate of its value comes out
        self.pi_net.train()    #Set the models to training mode
        self.V_net.train()
        self.pi_optimizer = optim.Adam(self.pi_net.parameters(), lr=self.config['alpha_pi'])    #each network gets its own optimizer and learning rate
        self.V_optimizer = optim.Adam(self.V_net.parameters(), lr=self.config['alpha_V'])

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def scale_state(self, s):
        '''A function to turn states into scaled inputs for the MLPs
            Takes:
                s -- a list of the six state values (or a list of such lists)
            Returns:
                a torch tensor of the state(s), scaled for the MLPs
        '''
        return torch.tensor(s).to(torch.float32)/self.scale    #scale the inputs

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def pi(self, s_t, greedy = False):
        '''A function to choose actions from the policy (the actor)
            Takes:
                s_t -- a list of the six state values
                greedy -- bool, True to take the most likely action instead of sampling one
            Returns:
                int, the index of the chosen action in A
                torch scalar, the log-probability of the chosen action (still attached to the actor so we can take its gradient)
                torch scalar, the entropy of the policy at s_t (how random the action choice is: 0 is deterministic, log(|A|) is uniform), also attached for the entropy bonus
        '''
        dist = Categorical(logits = self.pi_net(self.scale_state(s_t)))    #the softmax distribution over actions
        if greedy:
            a_idx = int(dist.probs.argmax())    #take the most likely action
        else:
            a_idx = int(dist.sample())    #draw an action according to its probability
        return a_idx, dist.log_prob(torch.tensor(a_idx)), dist.entropy()

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def V(self, s_t):
        '''A function to estimate the value of states (the critic)
            Takes:
                s_t -- a list of the six state values (or a list of such lists)
            Returns:
                torch tensor, the estimated value(s)
        '''
        return self.V_net(self.scale_state(s_t)).squeeze(-1)    #drop the trailing dimension of size 1

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def advantages_and_targets(self, states, rewards, s_next, bootstrap):
        '''A function to compute the advantages and critic targets for a rollout with generalized advantage estimation (GAE)
            Takes:
                states -- a list of the states visited at t = 0, 1, ..., n-1
                rewards -- a list of the rewards received at t = 1, 2, ..., n
                s_next -- the state at t = n, where the rollout stopped
                bootstrap -- bool, True if s_next is not terminal (the rollout hit n steps or the time limit), so the critic estimates what comes after it
            Returns:
                torch tensor, the advantage at each t -- the actor's learning signal
                torch tensor, the lambda-return at each t -- the critic's target
            lam sets the mix: lam = 1 gives the n-step return (everything to the end of the rollout, then the critic), lam = 0 gives the one-step TD error
        '''
        with torch.no_grad():    #advantages and targets are fixed labels, not something to take the gradient of
            values = self.V(states)    #the critic's estimates V(s_0), ..., V(s_{n-1})
            v_next = self.V(s_next) if bootstrap else torch.tensor(0.0)    #a win or a capture is worth nothing afterward; otherwise ask the critic
            next_values = torch.cat([values[1:], v_next.view(1)])    #V(s_1), ..., V(s_n)
            deltas = torch.tensor(rewards, dtype = torch.float32) + self.config['gamma']*next_values - values    #the one-step TD errors: how much better each single step went than the critic expected
            advantages = []
            A = torch.tensor(0.0)    #nothing is left to add after the last step of the rollout
            for delta in reversed(deltas):    #from the end of the rollout backward...
                A = delta + self.config['gamma']*self.config['lam']*A    #A_t = delta_t + gamma*lam*A_{t+1}
                advantages.insert(0, A)    #keep the advantages in time order
            advantages = torch.stack(advantages)
            targets = advantages + values    #the lambda-return: what the critic should have predicted
        return advantages, targets

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def update(self, states, log_probs, entropies, rewards, s_next, bootstrap):
        '''A function to take one A2C step using an n-step rollout
            Takes:
                states -- a list of the states visited at t = 0, 1, ..., n-1
                log_probs -- a list of the log-probabilities of the actions taken at t = 0, 1, ..., n-1
                entropies -- a list of the policy entropies at t = 0, 1, ..., n-1
                rewards -- a list of the rewards received at t = 1, 2, ..., n
                s_next -- the state at t = n, where the rollout stopped
                bootstrap -- bool, True if s_next is not terminal (the rollout hit n steps or the time limit), so the critic estimates what comes after it
            Returns:
                float, the critic loss
        '''
        advantages, targets = self.advantages_and_targets(states, rewards, s_next, bootstrap)    #the actor's learning signal and the critic's targets, from GAE
        V_s = self.V(states)    #the critic's current estimate of every state in the rollout (attached, for the critic loss)

        #the actor: push up the log-probability of actions with positive advantage, plus a bonus for keeping the policy random
        pi_loss = -(advantages*torch.stack(log_probs)).mean() - self.config['beta']*torch.stack(entropies).mean()    #minus because the optimizer descends
        self.pi_optimizer.zero_grad()    #zero out the gradients
        pi_loss.backward()    #compute gradients
        nn.utils.clip_grad_norm_(self.pi_net.parameters(), self.config['max_grad_norm'])    #keep any one rollout from moving the policy too far
        self.pi_optimizer.step()    #perform a single optimzation step

        #the critic: regress the state values onto the n-step targets
        V_loss = F.mse_loss(V_s, targets)
        self.V_optimizer.zero_grad()    #zero out the gradients
        V_loss.backward()    #compute gradients
        nn.utils.clip_grad_norm_(self.V_net.parameters(), self.config['max_grad_norm'])
        self.V_optimizer.step()    #perform a single optimzation step
        return V_loss.item()

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def render(self, state):
        '''A function to plot the prey (green) and predator (red)
            Takes:
                state -- the environment state dictionary
        '''
        x = [0,self.config['grid_size']-1]
        y = [0,self.config['grid_size']-1]
        plt.scatter(x, y,color='white')
        plt.plot(state['prey_location']['X'], state['prey_location']['Y'], 'o', color='green')
        plt.plot(state['predator_location']['X'], state['predator_location']['Y'], 'o', color='red')
        plt.show()


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#Set up the environments -- the chase and the agent
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
gamma = 0.9
env_config = {'grid_size':50
             ,'predator_start':None
             ,'prey_start':None
             ,'predator_radius':4
             ,'max_speed':5
             ,'max_steps':200    #the most turns an episode can last
             ,'gamma':gamma    #the discount factor, which potential-based shaping needs to match the agent's
             ,'base_weight':0.1    #the potential of being on base
             ,'predator_weight':0.3}    #the potential of being as far from the predator as possible (at opposite corners)

A = []    #init the actions
for v in [-1,0,1]:    #loop over vertical velo changes
    for h in [-1,0,1]:    #loop over horizonal velo changes
        A.append([v,h])    #record

agent_config = {'gamma':gamma    #the discount factor
                ,'alpha_pi':0.001    #the actor's learning rate
                ,'alpha_V':0.001    #the critic's learning rate
                ,'n_steps':5    #how many turns to collect before each update (fewer if the episode ends first)
                ,'lam':1    #the GAE weight: 1 uses the full n-step return, 0 uses only the one-step TD error, values in between trade variance for bias
                ,'beta':0.01    #the weight on the entropy bonus, which discourages the policy from becoming deterministic too early
                ,'max_grad_norm':0.5    #the largest gradient norm allowed in a single update
                ,'hidden_size':64    #the hidden layer size
                ,'A':A    #the action set
                ,'grid_size':env_config['grid_size']    #the size of the arena, used to scale the position inputs
                ,'max_speed':env_config['max_speed']    #the prey's max speed, used to scale the velocity inputs
               }   #set up the agent config

cenv = Chase_environment(env_config)    #set up the environment
prey = Agent(agent_config)    #set up the agent


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@
#Run the A2C algorithm
#@@@@@@@@@@@@@@@@@@@@@@@@@@@
N_eps = 20000    #specify the number of episodes
wins = []    #set up a recorder for whether the agent won or lost
returns = []    #set up a recorder for the discounted return of each episode
entropies = []    #set up a recorder for the average policy entropy of each episode
V_losses = []    #set up a recorder for the average critic loss of each episode

for epi in tqdm(range(N_eps)):    #loop over episodes
    s = cenv.get_state()    #the state at t = 0
    G = 0    #init the discounted return of the episode
    discount = 1    #init gamma**t
    ep_entropy = []    #init the policy entropy at each state visited this episode
    ep_V_loss = []    #init the critic loss of each update this episode
    done = False    #set the stopping condition
    while not done:    #while the episode goes on...
        states = []    #init the states visited this rollout
        log_probs = []    #init the log-probabilities of the actions taken this rollout
        rollout_entropies = []    #init the policy entropies this rollout
        rewards = []    #init the rewards received this rollout
        while len(rewards) < agent_config['n_steps'] and not done:    #collect up to n_steps turns
            a_idx, log_prob, entropy = prey.pi(s)    #sample an action from the policy for the state at t
            a = {'V':A[a_idx][0],'H':A[a_idx][1]}

            next_step = cenv.step(a)    #evolve the environment

            states.append(s)    #record the state at t
            log_probs.append(log_prob)    #record the log-probability of the action at t
            rollout_entropies.append(entropy)    #record the policy entropy at t
            rewards.append(next_step['reward'])    #record the reward at t+1
            G += discount*next_step['reward']    #add the discounted reward to the return
            discount *= agent_config['gamma']
            ep_entropy.append(float(entropy.detach()))    #a plain number for the learning curve
            s = cenv.get_state()    #the state at t+1
            done = next_step['done']    #record whether we've stopped

        #a win or a capture ends the chase for good, so nothing comes after it; running out of turns only stops the clock, so the critic still estimates what would come next
        terminated = next_step['win'] or cenv.check_caught()
        ep_V_loss.append(prey.update(states, log_probs, rollout_entropies, rewards, s, bootstrap = not terminated))    #update the actor and critic with the rollout

    returns.append(G)    #record the discounted return
    entropies.append(np.mean(ep_entropy))    #record the average policy entropy over the episode
    V_losses.append(np.mean(ep_V_loss))    #record the average critic loss over the episode
    wins.append(cenv.check_base())    #record whether the prey made it to base
    cenv.reset()    #reset the environment


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@
#Plot the learning curves
#@@@@@@@@@@@@@@@@@@@@@@@
window = 500    #the number of episodes to average over
smooth = lambda x: np.convolve(x, np.ones(window)/window, mode = 'valid')    #a rolling average
episodes = np.arange(window, len(wins) + 1)    #the episode at the end of each window

fig, ax = plt.subplots(2, 2, figsize = (14,8))
ax = ax.flatten()
ax[0].plot(episodes, smooth(np.array(wins, dtype = float)))    #the share of episodes won
ax[0].set_xlabel('Episode')
ax[0].set_ylabel('Win rate')
ax[0].set_title(f'Rolling win rate ({window} episodes)')
ax[1].plot(episodes, smooth(np.array(returns)))    #the average discounted return
ax[1].set_xlabel('Episode')
ax[1].set_ylabel('Discounted return (shaped)')
ax[1].set_title(f'Rolling return ({window} episodes)')
ax[2].plot(episodes, smooth(np.array(entropies)))    #the average policy entropy
ax[2].axhline(np.log(len(A)), color = 'gray', linestyle = '--', label = 'Uniform policy (max)')    #the entropy of choosing every action equally often
ax[2].set_ylim(bottom = 0)
ax[2].set_xlabel('Episode')
ax[2].set_ylabel('Policy entropy (nats)')
ax[2].set_title(f'Rolling policy entropy ({window} episodes)')
ax[2].legend(loc = 'lower left')
ax[3].plot(episodes, smooth(np.array(V_losses)))    #the average critic loss
ax[3].set_xlabel('Episode')
ax[3].set_ylabel('Critic loss (MSE)')
ax[3].set_title(f'Rolling critic loss ({window} episodes)')
plt.tight_layout()
plt.show()


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#Run A2C in inference mode
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
cenv = Chase_environment(env_config)    #set up the environment
N_eps = 10    #specify the number of episodes
for epi in range(N_eps):    #loop over episodes
    done = False    #set the stopping condition
    while not done:    #while the episode goes on...
        with torch.no_grad():    #no gradients are needed just to act
            a_idx, _, _ = prey.pi(cenv.get_state(), greedy = False)    #sample an action from the learned policy (greedy = True takes the most likely action instead)
        a = {'V':A[a_idx][0],'H':A[a_idx][1]}
        next_step = cenv.step(a)    #evolve the environment
        done = next_step['done']    #record whether we've stopped
        prey.render(cenv.state)
        time.sleep(0.1)
        clear_output(wait=True)

    cenv.reset()    #reset the environment
